# 04. Model Training & Baseline Benchmarking

## Overview
This notebook trains baseline regressor architectures (**RandomForestRegressor**, **GradientBoostingRegressor**, **LinearRegression**) for predicting system power output ($	ext{kW}$) and performs 5-fold cross-validation and hyperparameter optimization.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("Scikit-Learn Machine Learning Stack Initialized!")

## 1. Synthetic Dataset Generation for Ground-Truth Training

In [ ]:
np.random.seed(42)
n = 2000

solar_ghi = np.random.uniform(3.5, 7.2, n)
wind_speed = np.random.uniform(3.0, 10.5, n)
slope = np.random.exponential(2.5, n).clip(0, 15)
temp = np.random.uniform(15, 42, n)
capacity_mw = np.random.choice([10, 25, 50, 100], n)

# Target Generation Formula with physical constraints + noise
base_yield = (solar_ghi * 420.0 + wind_speed ** 2.2 * 85.0) * capacity_mw
slope_penalty = 1.0 - (slope * 0.02)
temp_penalty = 1.0 - ((temp - 25).clip(lower=0) * 0.004)
noise = np.random.normal(0, 250.0, n)

target_power_kw = (base_yield * slope_penalty * temp_penalty + noise).clip(lower=0)

X = pd.DataFrame({
    'solar_ghi': solar_ghi,
    'wind_speed': wind_speed,
    'slope_deg': slope,
    'temp_ambient': temp,
    'capacity_mw': capacity_mw
})
y = target_power_kw

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape}, Test Set: {X_test.shape}")

## 2. Train Regressor Architectures & Cross-Validation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results.append({
        'Model': name,
        'MAE (kW)': round(mae, 2),
        'RMSE (kW)': round(rmse, 2),
        'R2 Score': round(r2, 4)
    })

pd.DataFrame(results)

## 3. Hyperparameter Tuning for Production Random Forest

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best R2 Score: {grid_search.best_score_:.4f}")

## 4. Serialize Production Artifact

In [ ]:
joblib.dump(best_rf, '../models/power_forecaster.joblib')
print("Model successfully serialized to ../models/power_forecaster.joblib")